# BASELINE LOGISTIC REGRESSION MODEL (V0)

In [14]:
from pathlib import Path 
import os 
import warnings 

import numpy as np 
import pandas as pd 
from datasets import load_dataset 


In [ ]:
#Load Data from HF 

markets = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "markets"
)
prices = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "prices"
)

spot_prices = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "spot_prices"
)

orderbook = load_dataset(
    "aliplayer1/polymarket-crypto-updown",
    "orderbook",
    streaming=True 
)

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1408 [00:00<?, ?it/s]

data/orderbook/crypto=BNB/timeframe=1-ho(…): reconstructing file:   0%|          |  0.00B /  194MB            

data/orderbook/crypto=BNB/timeframe=1-ho(…): downloading bytes:           |  0.00B            

data/orderbook/crypto=BNB/timeframe=1-ho(…): reconstructing file:   0%|          |  0.00B / 27.5kB            

...

data/orderbook/crypto=BNB/timeframe=4-ho(…): reconstructing file:   0%|          |  0.00B /  166MB            

data/orderbook/crypto=BNB/timeframe=4-ho(…): downloading bytes:           |  0.00B            

data/orderbook/crypto=BNB/timeframe=4-ho(…): reconstructing file:   0%|          |  0.00B / 31.9kB            

data/orderbook/crypto=BNB/timeframe=4-ho(…): downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [16]:
# Load Data from Markets/Prices/Spot / Filter Data to access BTC 5m + create columns for prediction_ts and the corresponding Spot Price 


# MARKETS 

df_markets = markets['train'].to_pandas() 
df_markets_btc5m = df_markets.loc[(df_markets['timeframe'] == '5-minute') & (df_markets['crypto'] == 'BTC')]

WINDOW = 300 # represents 5m

## start_ts : polymarket market open / created (is before window start)
## window_start_ts : start of 5m resolution window 
## end_ts : end of 5m resolution window 

df_markets_btc5m["window_start_ts"] = (
    df_markets_btc5m["end_ts"] - WINDOW
)


marketid_5m = set(df_markets_btc5m['market_id'])

# PRICES 

prices_btc5m = prices["train"].filter(
    lambda batch: [
        market_id in marketid_5m
        for market_id in batch['market_id']
    ],
    batched=True
)

df_prices_btc5m = prices_btc5m.to_pandas() 

# SPOT PRICES 

df_spot_prices = spot_prices['train'].to_pandas() 

df_spot_prices_btc = df_spot_prices.loc[(df_spot_prices['symbol'] == 'btc/usd') & (df_spot_prices['source'] == 'chainlink_proxy')]




In [ ]:
# Filter df_prices_btc5m to only include tiemstamps for prediction_ts (t-1m, t-1:30m, t-2m, previous 5m market resolution) 

T1 = 60 
T1_30 = 90 
T2 = 120 

## Prediction ts for every 5m BTC market (n denotes prediction_ts time)

markets_tn =  df_markets_btc5m.copy() 
markets_tn['prediction_ts'] = markets_tn

## Attach market start time to each price observation 


